# 🎙️ Estúdio de Voz Oficial do Mascote FEFO (Coqui XTTS-v2)
**Notebook de Alta Fidelidade** — Configurado para clonar e sintetizar a voz oficial do FEFO em **Português do Brasil** nativo com o motor **XTTS-v2**.

---
### ⚡ Como Usar (Apenas 1 Clique):
1. Verifique a GPU no menu: **Ambiente de execução** ➔ **Alterar tipo de ambiente de execução** ➔ Selecione **T4 GPU** ➔ **Salvar**.
2. Clique em **Ambiente de execução** ➔ **Executar tudo** (ou atalho `Ctrl + F9`).
3. Em cerca de 2 minutos, o modelo estará carregado, gerará o primeiro áudio para você ouvir e abrirá um painel visual completo com link compartilhável!

In [ ]:
# 1. Instalação Otimizada do Coqui XTTS-v2 e Dependências
import os, torch
os.environ['COQUI_TOS_AGREED'] = '1'

print('=' * 60)
if torch.cuda.is_available():
    print(f'✓ GPU DETECTADA: {torch.cuda.get_device_name(0)}')
    print(f'✓ VRAM Disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ ATENÇÃO: Nenhuma GPU detectada! Ative em: Ambiente de execução -> Alterar tipo -> T4 GPU')
print('=' * 60)

print('[*] Instalando o Coqui XTTS-v2 e dependências de áudio...')
!pip install -q coqui-tts transformers==4.36.2 accelerate gradio soundfile pydub
print('✓ Instalação concluída com sucesso!')


In [ ]:
# 2. Baixar Repositório do FEFO e Carregar o Modelo XTTS-v2 na GPU
import os
from TTS.api import TTS

%cd /content
if not os.path.exists('/content/fefo-repo'):
    print('[*] Baixando materiais e áudios oficiais do repositório FEFO...')
    !git clone https://github.com/elCortelini/fefo-v1.git /content/fefo-repo

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[*] Carregando o modelo neural XTTS-v2 no acelerador {device.upper()}...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('✓ Modelo XTTS-v2 carregado com sucesso!')


In [ ]:
# 3. Teste Imediato com Player de Áudio do FEFO
import os, glob, IPython.display as ipd

# Identificar o áudio oficial de referência
ref_audio = '/content/fefo-repo/repository/ficofelizporvc.wav'
if not os.path.exists(ref_audio):
    candidatos = glob.glob('/content/fefo-repo/**/ficofelizporvc.wav', recursive=True)
    ref_audio = candidatos[0] if candidatos else '/content/fefo-repo/repository/audio/a0001.wav'

print(f'✓ Áudio de referência oficial do FEFO: {ref_audio}')

frase_teste = 'Olá amiguinho! Eu sou o Fefo, vamos brincar juntos hoje? Estou muito feliz em te ver!'
arquivo_saida = '/content/teste_fefo_xtts.wav'

print('[*] Gerando fala em Português do Brasil com a voz clonada do FEFO...')
tts.tts_to_file(
    text=frase_teste,
    file_path=arquivo_saida,
    speaker_wav=ref_audio,
    language='pt',
    temperature=0.7,
    repetition_penalty=2.0
)

print('=' * 60)
print('🎉 ÁUDIO DO FEFO GERADO COM SUCESSO! CLIQUE NO PLAY ABAIXO PARA OUVIR:')
print('=' * 60)
display(ipd.Audio(arquivo_saida))


In [ ]:
# 4. Painel Visual WebUI do FEFO (Gradio com Link Público)
import os, glob, gradio as gr

# Coletar vozes de referência do FEFO
vozes_disponiveis = {}
candidatos_base = glob.glob('/content/fefo-repo/**/*.wav', recursive=True) + glob.glob('/content/fefo-repo/**/*.mp3', recursive=True)
for c in candidatos_base:
    bn = os.path.basename(c)
    if any(k in bn.lower() for k in ['ficofeliz', 'voz base', 'fefo_0001', 'a0001']):
        vozes_disponiveis[f'Voz Oficial: {bn}'] = c

if not vozes_disponiveis:
    vozes_disponiveis['Voz Padrão FEFO'] = ref_audio

def sintetizar_voz_fefo(texto, audio_ref, velocidade, temperatura):
    if not texto.strip():
        return None
    saida = '/content/fefo_webui_saida.wav'
    tts.tts_to_file(
        text=texto.strip(),
        file_path=saida,
        speaker_wav=audio_ref,
        language='pt',
        speed=float(velocidade),
        temperature=float(temperatura)
    )
    return saida

with gr.Blocks(title='🧸 Estúdio de Voz do Mascote FEFO (XTTS-v2)') as app:
    gr.Markdown('# 🎙️ Estúdio de Voz Oficial do Mascote FEFO')
    gr.Markdown('Digite qualquer frase em Português e gere a fala com o timbre e entonação do FEFO.')
    
    with gr.Row():
        with gr.Column():
            txt_in = gr.Textbox(
                label='Texto para o FEFO falar:',
                value='Olá! Que bom ver você aqui no nosso aplicativo!',
                lines=3
            )
            voz_in = gr.Dropdown(
                label='Áudio Base de Referência (Timbre do FEFO):',
                choices=list(vozes_disponiveis.values()),
                value=list(vozes_disponiveis.values())[0]
            )
            with gr.Row():
                spd_in = gr.Slider(0.7, 1.4, value=1.0, step=0.05, label='Velocidade da Fala')
                temp_in = gr.Slider(0.1, 1.0, value=0.65, step=0.05, label='Expressividade')
            
            btn = gr.Button('✨ Gerar Áudio do FEFO', variant='primary')
        
        with gr.Column():
            audio_out = gr.Audio(label='Áudio Gerado do FEFO', type='filepath')
    
    btn.click(
        fn=sintetizar_voz_fefo,
        inputs=[txt_in, voz_in, spd_in, temp_in],
        outputs=[audio_out]
    )

print('=' * 60)
print('🚀 INICIANDO INTERFACE VISUAL DO FEFO...')
print('=' * 60)
app.launch(share=True, debug=False)
